# COMEPHORE Ingestion & Daily Processing

Downloads hourly radar-gauge precipitation from data.gouv.fr,
aggregates to daily with error propagation, saves per-month NetCDF.

**Storage strategy:**
- TAR archives → `data/raw/comephore/archives/` (keep on external disk)
- Extracted hourly GeoTIFFs → `data/raw/comephore/hourly/` (temporary, delete after processing)
- Daily NetCDF → `data/processed/comephore/comephore_daily_YYYYMM.nc` (~50–100 MB/month)

**Variables in daily output:**
| Variable | Unit | Aggregation |
|----------|------|-------------|
| `precip_mm` | mm/day | sum(hourly RR) / 100 |
| `precip_error_mm` | mm/day | sqrt(sum(hourly ERR²)) / 100 |
| `quality_frac` | 0–1 | mean(hourly QUALIF) / 100 |
| `n_hours_valid` | 0–24 | count of non-NA hours |

In [1]:
import os
import logging
from datetime import date
from pathlib import Path

REPO_ROOT = os.path.dirname(os.getcwd())
os.chdir(REPO_ROOT)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    datefmt="%H:%M:%S",
)

## Option A — Full pipeline (recommended)

Downloads, extracts, and processes in one call. Set `cleanup_hourly=True`
to delete the extracted GeoTIFFs after each month (TARs are kept).

In [2]:
from irrigator.ingestion.comephore_client import run_comephore_pipeline

# Process one year at a time — easy to resume if interrupted
daily_paths = run_comephore_pipeline(
    start=date(2021, 5, 1),
    end=date(2026, 4, 30),
    cleanup_hourly=True,   # delete hourly GeoTIFFs after aggregation
    overwrite_daily=False,  # skip months already processed
)

print(f"\nProduced {len(daily_paths)} monthly files:")
for p in daily_paths:
    print(f"  {p.name}  ({p.stat().st_size / 1e6:.1f} MB)")

09:42:21 [numexpr.utils] INFO: Note: NumExpr detected 22 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
09:42:21 [numexpr.utils] INFO: NumExpr defaulting to 16 threads.
09:42:21 [irrigator.ingestion.comephore_client] INFO: [2021-05] Step 1/3: Downloading...
09:42:21 [irrigator.ingestion.comephore_client] INFO: Downloading COMEPHORE 2021-05 to data/raw/comephore/archives/H_COMEPHORE_202105.tar
09:42:30 [irrigator.ingestion.comephore_client] INFO: Saved COMEPHORE archive: data/raw/comephore/archives/H_COMEPHORE_202105.tar (274.7 MB)
09:42:30 [irrigator.ingestion.comephore_client] INFO: [2021-05] Step 2/3: Extracting...
09:42:30 [irrigator.ingestion.comephore_client] INFO: [2021-05] Step 3/3: Aggregating to daily...
09:42:30 [irrigator.ingestion.comephore_client] INFO: Processing COMEPHORE 2021-05 to daily (2021-05-01 → 2021-05-31)
09:42:30 [irrigator.ingestion.comephore_client] ERROR: [2021-05] Failed: No COMEPHORE RR GeoTIFF files in data/raw/comephore/hourly be

KeyboardInterrupt: 

## Option B — Step by step (for debugging)

Run each step manually to inspect intermediate results.

In [ ]:
from irrigator.ingestion.comephore_client import (
    fetch_comephore_range,
    extract_comephore_archive,
    open_comephore_hourly,
    process_comephore_month_to_daily,
)

# Step 1: Download monthly TARs
archives = fetch_comephore_range(date(2026, 3, 1), date(2026, 3, 1))
print(f"Downloaded {len(archives)} archives")

# Step 2: Extract
for archive in archives:
    extract_comephore_archive(archive)

In [ ]:
# Step 3: Inspect hourly data before aggregation
accum = open_comephore_hourly(var_type="accum", start=date(2026, 3, 1), end=date(2026, 3, 1))
error = open_comephore_hourly(var_type="error", start=date(2026, 3, 1), end=date(2026, 3, 1))

print(f"Hourly accum: {dict(accum.sizes)}")
print(f"Hourly error: {dict(error.sizes)}")
print(f"Accum range: [{float(accum.min()):.0f}, {float(accum.max()):.0f}] (raw units, /100 for mm)")
print(f"Error range: [{float(error.min()):.0f}, {float(error.max()):.0f}] (raw units, /100 for mm)")

In [ ]:
# Step 4: Process to daily
daily_path = process_comephore_month_to_daily(2026, 3, overwrite=True)
print(f"Saved: {daily_path}")

## Inspect daily output

In [ ]:
import xarray as xr

ds = xr.open_dataset(daily_path)
ds

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Pick a sample day
sample = ds.isel(time=0)

sample.precip_mm.plot(ax=axes[0, 0], cmap="Blues", vmin=0)
axes[0, 0].set_title(f"Precipitation [mm/day] — {str(sample.time.values)[:10]}")

sample.precip_error_mm.plot(ax=axes[0, 1], cmap="Oranges", vmin=0)
axes[0, 1].set_title("Propagated error [mm/day]")

sample.quality_frac.plot(ax=axes[1, 0], cmap="RdYlGn", vmin=0, vmax=1)
axes[1, 0].set_title("Quality fraction [0–1]")

sample.n_hours_valid.plot(ax=axes[1, 1], cmap="viridis", vmin=0, vmax=24)
axes[1, 1].set_title("Valid hours [0–24]")

plt.tight_layout()
plt.show()

## Load processed archive for downstream use

In [ ]:
from irrigator.ingestion.comephore_client import load_comephore_daily

# Load all available daily COMEPHORE
ds_all = load_comephore_daily()
print(f"Daily COMEPHORE: {dict(ds_all.sizes)}")
print(f"Date range: {str(ds_all.time.values[0])[:10]} → {str(ds_all.time.values[-1])[:10]}")

## Batch processing — year by year

For building the full 1997–2026 archive, process one year at a time.
After each year, move the TAR archives to external storage.

In [ ]:
# Uncomment to run year by year:
# for year in range(1997, 2027):
#     print(f"\n{'='*60}")
#     print(f"Processing {year}")
#     print(f"{'='*60}")
#     daily_paths = run_comephore_pipeline(
#         start=date(year, 1, 1),
#         end=date(year, 12, 1),
#         cleanup_hourly=True,
#         overwrite_daily=False,
#     )
#     print(f"  → {len(daily_paths)} months processed")
#     # After this, move data/raw/comephore/archives/*{year}* to external disk